In [8]:
import os 
import sys
from pathlib import Path
from dotenv import load_dotenv

In [9]:
load_dotenv(Path.cwd().parent / ".env")

True

In [3]:
repo_root = Path.cwd().parent
sys.path.insert(0, str(repo_root))

In [4]:
from ingestion.jolpica.helper import logging_setup
from ingestion.jolpica.race_results import get_race_results
from ingestion.jolpica.writer import df_write_to_s3_raw


In [5]:
logging_setup()
df = get_race_results(2026)
print(df.shape)
print(df.head(2))

[2026-03-18 10:19:13] INFO [ingestion.jolpica.race_results] Getting race result data for 2026
[2026-03-18 10:19:14] INFO [ingestion.jolpica.helper] 2026/results: offset 0/44
[2026-03-18 10:19:14] INFO [ingestion.jolpica.race_results] Race results 2026: 44 rows across 2 rounds


(44, 14)
   season  round              race_name                         circuit  \
0    2026      1  Australian Grand Prix  Albert Park Grand Prix Circuit   
1    2026      1  Australian Grand Prix  Albert Park Grand Prix Circuit   

         date  driver_id driver_code            driver_name constructor  grid  \
0  2026-03-08    russell         RUS         George Russell    Mercedes     1   
1  2026-03-08  antonelli         ANT  Andrea Kimi Antonelli    Mercedes     2   

   position  points    status  fastest_lap_rank  
0         1    25.0  Finished               6.0  
1         2    18.0  Finished               3.0  


In [6]:
s3_path = df_write_to_s3_raw(df=df, source="jolpica", year="2026", endpoint="race_results")
print(f"Written to: {s3_path}")

Written to: s3://f1-raw-395249043157/jolpica/2026/race_results.parquet


In [13]:
import boto3
s3 = boto3.client("s3", region_name=os.getenv("AWS_REGION"))
bucket = os.getenv("S3_BUCKET_RAW")
Key="jolpica/2026/race_results.parquet"
response = s3.get_object(Bucket=bucket, Key=Key)
content = response["ContentLength"]
print(content)

11187


In [14]:
import pandas as pd
import s3fs

fs = s3fs.S3FileSystem(
    key=os.getenv("AWS_ACCESS_KEY_ID"),
    secret=os.getenv("AWS_SECRET_ACCESS_KEY"),
    client_kwargs={"region_name": os.getenv("AWS_REGION")},
)

bucket = os.getenv("S3_BUCKET_RAW")

with fs.open(f"s3://{bucket}/jolpica/2026/race_results.parquet", "rb") as f:
    df = pd.read_parquet(f)

print(df.head(5))

   season  round              race_name                         circuit  \
0    2026      1  Australian Grand Prix  Albert Park Grand Prix Circuit   
1    2026      1  Australian Grand Prix  Albert Park Grand Prix Circuit   
2    2026      1  Australian Grand Prix  Albert Park Grand Prix Circuit   
3    2026      1  Australian Grand Prix  Albert Park Grand Prix Circuit   
4    2026      1  Australian Grand Prix  Albert Park Grand Prix Circuit   

         date  driver_id driver_code            driver_name constructor  grid  \
0  2026-03-08    russell         RUS         George Russell    Mercedes     1   
1  2026-03-08  antonelli         ANT  Andrea Kimi Antonelli    Mercedes     2   
2  2026-03-08    leclerc         LEC        Charles Leclerc     Ferrari     4   
3  2026-03-08   hamilton         HAM         Lewis Hamilton     Ferrari     7   
4  2026-03-08     norris         NOR           Lando Norris     McLaren     6   

   position  points    status  fastest_lap_rank   source  \
0 